In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/output/olist_integrated_dataset.csv"
)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

df.head()

Rows: 112,650
Columns: 59


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,customer_geo_state,seller_geo_lat,seller_geo_lng,seller_geo_city,seller_geo_state,item_total_value,delivery_days,delivery_vs_estimate_days,delivery_performance,customer_seller_distance_km
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,RJ,-22.499217,-44.125997,volta redonda,RJ,72.19,7.614421,-8.011250,On or before estimate,301.299537
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,SP,-23.565803,-46.519192,sao paulo,SP,259.83,16.216181,-2.330278,On or before estimate,588.544256
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,MG,-22.272310,-46.165743,borda da mata,MG,216.87,7.948437,-13.444954,On or before estimate,312.955217
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,SP,-20.554801,-47.387749,franca,SP,25.78,6.147269,-5.435660,On or before estimate,295.319582
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,SP,-22.930512,-53.136361,loanda,PR,218.04,25.114352,-15.303808,On or before estimate,646.459821


In [3]:
print(df.columns.tolist())

['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'dq_missing_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'dq_invalid_weight', 'product_category_name_english', 'dq_missing_category_translation', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'total_payment_value', 'payment_record_count', 'payment_type_count', 'max_payment_installments', 'payment_types', 'dq_missing_payment', 'review_count', 'unique_review_count', 'average_review_score', 'min_review_score', 'max_review_score', 'reviews_with_comments', 'dq_missing

In [4]:
# Slide 11 - Key Business KPIs

total_orders = df["order_id"].nunique()

sales_value = df["item_total_value"].sum()

avg_delivery = df["delivery_days"].mean()

avg_review = df["average_review_score"].mean()

print(f"Total Orders: {total_orders:,}")
print(f"Sales Value: R$ {sales_value:,.2f}")
print(f"Average Delivery: {avg_delivery:.1f} days")
print(f"Average Review Score: {avg_review:.2f} / 5")

Total Orders: 98,666
Sales Value: R$ 15,843,553.24
Average Delivery: 12.5 days
Average Review Score: 4.03 / 5


In [5]:
top_categories = (
    df.groupby("product_category_name_english")["item_total_value"]
      .sum()
      .sort_values(ascending=False)
      .head(4)
)

print(top_categories)

product_category_name_english
health_beauty     1441248.07
watches_gifts     1305541.61
bed_bath_table    1241681.72
sports_leisure    1156656.48
Name: item_total_value, dtype: float64


In [6]:
delivery_review = (
    df.dropna(subset=["delivery_days", "average_review_score"])
      .groupby("average_review_score")["delivery_days"]
      .mean()
)

print(delivery_review)

average_review_score
1.000000    19.620337
1.500000    14.874045
2.000000    15.816367
2.500000    16.432744
3.000000    14.029473
3.333333    22.033148
3.500000    12.802536
4.000000    12.209502
4.333333     3.765359
4.500000    12.974410
5.000000    10.646918
Name: delivery_days, dtype: float64


In [7]:
delivery_review_plot = (
    df.dropna(subset=["delivery_days", "average_review_score"])
      .assign(
          review_score=lambda x: x["average_review_score"].round()
      )
      .groupby("review_score")["delivery_days"]
      .mean()
      .reset_index()
)

print(delivery_review_plot)

   review_score  delivery_days
0           1.0      19.620337
1           2.0      15.818601
2           3.0      14.030343
3           4.0      12.212341
4           5.0      10.646918


In [8]:
distance_freight = (
    df.dropna(subset=["customer_seller_distance_km", "freight_value"])
      .assign(
          distance_band=lambda x: pd.cut(
              x["customer_seller_distance_km"],
              bins=[0, 250, 500, 1000, 2000, float("inf")],
              labels=["0–250", "251–500", "501–1,000", "1,001–2,000", "2,000+"]
          )
      )
      .groupby("distance_band", observed=True)["freight_value"]
      .mean()
)

print(distance_freight)

distance_band
0–250          13.165144
251–500        19.255351
501–1,000      21.144873
1,001–2,000    29.201295
2,000+         35.915433
Name: freight_value, dtype: float64
